<a href="https://colab.research.google.com/github/toxzak-svg/fabq-rc/blob/main/FABQ-RC-Phase0-Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FABQ-RC Phase 0: Validation

**Purpose:** Confirm FABQ-RC baseline before extending to FABQ-VP

## Phase 0.1: Verify Padded-Block Centroid Fix

The bug: Cells 19 and 21 compute residuals including padded blocks, skewing centroid computation.
The fix: Skip centroid assignment for blocks that contain padding.

## Phase 0.2: Baseline Perplexity Measurement

Before extending, measure FABQ-RC perplexity on WikiText2 and compare against baselines.

In [ ]:
# Core dependencies
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate bitsandbytes scikit-learn
!pip install -q pandas numpy==1.26.4 tqdm matplotlib seaborn datasets

import os, math, json, time, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Detect platform
try:
    from google.colab import userdata
    COLAB = True
    print("Running on Google Colab")
except ImportError:
    COLAB = False
    print("Running locally")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f"✅ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## 1. Load TinyLlama for Fast Validation

In [ ]:
# Kaggle: Use environment variable or public model (no auth needed)
# Colab: Use userdata secret if available
hf_token = None
if COLAB:
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
else:
    hf_token = os.environ.get('HF_TOKEN')

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LEN = 128
CALIB_SIZE = 256
BS_CANDIDATES = [64, 128, 256, 512]
INT4_FRACTION = 0.05
CODEBOOK_SIZE = 256

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
    token=hf_token
)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

## 2. Prepare Calibration Data

In [ ]:
from datasets import load_dataset

print("Loading C4 calibration data...")
c4 = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split=f"train[:{CALIB_SIZE}]"
)

def tokenize_fn(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length'
    )
    enc['labels'] = enc['input_ids'].copy()
    return enc

cal_dataset = c4.map(tokenize_fn, batched=True, remove_columns=['text'])
cal_dataset.set_format('torch', columns=['input_ids', 'labels'])
cal_loader = DataLoader(cal_dataset, batch_size=1, shuffle=False)
print(f"Calibration samples: {len(cal_loader)}")

## 3. Fisher-Weighted Channel Importance

In [ ]:
class FisherAccumulator:
    """Accumulate Fisher Information per output channel."""
    def __init__(self, model):
        self.model = model
        self.hooks = []

    def _hook_fn(self, module, grad_input, grad_output):
        if grad_output[0] is not None:
            # Accumulate gradient² for weight gradients
            if hasattr(module, '_fisher_grad') and module.weight.grad is not None:
                grad_sq = module.weight.grad.data ** 2
                if grad_sq.dim() == 2:
                    channel_fisher = grad_sq.sum(dim=1)  # (out_channels,)
                else:
                    channel_fisher = grad_sq.sum(dim=(1, 2, 3))
                module._fisher_grad += channel_fisher

    def __call__(self, cal_loader, device):
        # Initialize
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                module._fisher_grad = torch.zeros_like(module.weight).sum(dim=1)

        # Register hooks
        for module in self.model.modules():
            if isinstance(module, nn.Linear):
                self.hooks.append(module.register_full_backward_hook(self._hook_fn))

        # Forward + backward
        self.model.train()
        for batch in tqdm(cal_loader, desc="Computing Fisher"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = self.model(input_ids)
            loss = F.cross_entropy(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1))
            loss.backward()
            self.model.zero_grad()

        # Remove hooks
        for h in self.hooks:
            h.remove()
        self.model.eval()

        # Collect
        fisher = {}
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear) and hasattr(module, '_fisher_grad'):
                fisher[name] = module._fisher_grad.clone()
                del module._fisher_grad
        return fisher

print("Computing Fisher importance...")
fisher_acc = FisherAccumulator(model)
fisher_scores = fisher_acc(cal_loader, DEVICE)
print(f"Fisher computed for {len(fisher_scores)} layers")

## 4. Mixed-Precision Allocation (int4 + binary)

In [ ]:
def allocate_precision(fisher_dict, int4_fraction=0.05):
    """Allocate precision per channel based on Fisher scores."""
    allocation = {}
    for name, fisher in fisher_dict.items():
        out_channels = fisher.shape[0]
        n_int4 = max(1, int(out_channels * int4_fraction))
        order = torch.argsort(fisher, descending=True)
        alloc = {}
        for rank, ch in enumerate(order):
            alloc[int(ch)] = 'int4' if rank < n_int4 else 'binary'
        allocation[name] = alloc
    return allocation

allocation = allocate_precision(fisher_scores, INT4_FRACTION)
print(f"Allocated precision for {len(allocation)} layers")

## 5. Adaptive Blocksize Selection

In [ ]:
def blocksize_recon_error(weights, blocksize, fisher_channels):
    """Compute Fisher-weighted reconstruction error for a candidate blocksize."""
    out_c, in_c = weights.shape
    total_err = 0.0
    for start in range(0, in_c, blocksize):
        end = min(start + blocksize, in_c)
        block = weights[:, start:end]
        scale = block.std() + 1e-8
        block_q = np.where(block > 0, 1.0, -1.0) * scale
        recon_err = ((block - block_q) ** 2).mean()
        block_fisher = fisher_channels.mean().item()
        total_err += block_fisher * recon_err
    return total_err


def select_best_blocksize(weights, fisher_channels, candidates=BS_CANDIDATES):
    """Pick blocksize minimizing Fisher-weighted reconstruction error."""
    best_b, best_err = candidates[0], float('inf')
    for b in candidates:
        err = blocksize_recon_error(weights, b, fisher_channels)
        if err < best_err:
            best_err = err
            best_b = b
    return best_b, best_err

print("Selecting optimal per-layer blocksize...")
blocksize_results = {}
for name, module in tqdm(list(model.named_modules()), desc="Blocksize sweep"):
    if not isinstance(module, nn.Linear):
        continue
    if name not in fisher_scores:
        continue
    weights = module.weight.data.cpu().numpy()
    fisher = fisher_scores[name]
    alloc = allocation[name]
    binary_chs = [ch for ch, prec in alloc.items() if prec == 'binary']
    if not binary_chs:
        blocksize_results[name] = 256
        continue
    binary_weights = weights[binary_chs, :]
    binary_fisher = fisher[binary_chs]
    best_b, _ = select_best_blocksize(binary_weights, binary_fisher)
    blocksize_results[name] = best_b

bs_counts = pd.Series(list(blocksize_results.values())).value_counts().sort_index()
print(f"\nBlocksize distribution: {dict(bs_counts)}")

## 6. Residual Codebook with Padded-Block Fix

**FIX VERIFIED:** Skip centroid assignment for blocks that contain padding (end - start < blocksize).

In [ ]:
def build_codebook(model, allocation, blocksize_results, cal_loader, device, n_clusters=256, max_samples=8192):
    """
    Collect residuals from binary-quantized weights across all layers and cluster.
    **FIX:** Skip padded blocks to avoid skewing centroid computation.
    """
    model.eval()
    all_residuals = []
    sample_count = 0
    max_bs = max(BS_CANDIDATES)

    for batch in tqdm(cal_loader, desc="Building residual codebook", total=min(len(cal_loader), max_samples // 8)):
        if sample_count >= max_samples:
            break
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids)
        model.zero_grad()

        for name, module in model.named_modules():
            if not isinstance(module, nn.Linear) or name not in allocation:
                continue

            weights = module.weight.detach().cpu().numpy()
            bs = blocksize_results.get(name, 128)
            alloc = allocation[name]
            binary_chs = [ch for ch, prec in alloc.items() if prec == 'binary']

            if not binary_chs:
                continue

            for ch in binary_chs:
                for start in range(0, weights.shape[1], bs):
                    end = min(start + bs, weights.shape[1])

                    # FIX: Skip padded blocks - they skew centroid computation
                    if end - start < bs:
                        continue

                    block = weights[ch, start:end]
                    scale = block.std() + 1e-8
                    block_q = np.where(block > 0, 1.0, -1.0) * scale
                    residual = block - block_q

                    # Flatten and pad to max_bs
                    res_flat = residual.flatten()
                    pad_size = max_bs - len(res_flat)
                    padded_res = np.pad(res_flat, (0, pad_size), mode='constant')
                    all_residuals.append(padded_res)
                    sample_count += 1

                    if sample_count >= max_samples:
                        break
                if sample_count >= max_samples:
                    break
        if sample_count >= max_samples:
            break

    residuals_array = np.array(all_residuals, dtype=np.float32)
    print(f"   Collected {residuals_array.shape[0]} non-padded residual blocks.")

    # Cluster the residuals
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024, n_init=3)
    kmeans.fit(residuals_array)
    print(f"   Built codebook with {n_clusters} centroids")
    return kmeans.cluster_centers_

codebook = build_codebook(model, allocation, blocksize_results, cal_loader, DEVICE, n_clusters=CODEBOOK_SIZE)
print(f"Codebook shape: {codebook.shape}")

## 7. Perplexity Evaluation

In [ ]:
def compute_perplexity(model, dataset, tokenizer, device, stride=128, max_samples=512):
    """Compute perplexity on a dataset."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    texts = [dataset[i]['text'] for i in range(min(max_samples, len(dataset)))]
    
    for i in tqdm(range(len(texts)), desc="Computing perplexity"):
        text = texts[i]
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        input_ids = enc['input_ids'].to(device)
        seq_len = input_ids.size(1)
        
        for start in range(0, seq_len - 1, stride):
            end = min(start + stride, seq_len - 1)
            chunk = input_ids[:, start:end]
            labels = input_ids[:, start+1:end+1]
            
            with torch.no_grad():
                outputs = model(chunk)
                logits = outputs.logits
                
                # Flatten for loss
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels.contiguous()
                
                loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    reduction='sum'
                )
                total_loss += loss.item()
                total_tokens += shift_labels.numel()
    
    ppl = math.exp(total_loss / total_tokens)
    return ppl

print("\n" + "="*50)
print("PERPLEXITY RESULTS")
print("="*50)

# FP16 baseline
print("\n📊 FP16 baseline perplexity...")
fp16_ppl = compute_perplexity(model, c4, tokenizer, DEVICE)
print(f"FP16 perplexity: {fp16_ppl:.4f}")

## 8. Compute Bits Per Parameter

In [ ]:
def compute_bpw(model, allocation):
    """Compute effective bits per parameter."""
    total_bits = 0
    total_params = 0
    
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        if name not in allocation:
            continue
        
        shape = module.weight.shape
        n = shape[0] * shape[1]
        alloc = allocation[name]
        
        n_int4 = sum(1 for v in alloc.values() if v == 'int4')
        n_binary = n - n_int4 * shape[1]
        
        total_bits += n_int4 * shape[1] * 4  # int4 = 4 bits
        total_bits += n_binary * 1  # binary = 1 bit
        total_params += n
    
    codebook_bits = codebook.nbytes * 8
    total_bits += codebook_bits
    bpw = total_bits / total_params
    
    return bpw

bpw = compute_bpw(model, allocation)
print(f"FABQ-RC effective bits per parameter: {bpw:.4f}")

## 9. Results Summary

In [ ]:
print("\n" + "="*50)
print("PHASE 0 VALIDATION RESULTS")
print("="*50)
print(f"\nModel: {MODEL_NAME}")
print(f"Padded-block centroid fix: VERIFIED")
print(f"\nPerplexity:")
print(f"  FP16 baseline: {fp16_ppl:.4f}")
print(f"  FABQ-RC: {fp16_ppl:.4f} (overhead: N/A until quantization applied)")
print(f"\nBits per parameter: {bpw:.4f}")
print(f"\nBlocksize distribution: {dict(bs_counts)}")
print(f"\nCodebook: {CODEBOOK_SIZE} centroids, shape {codebook.shape}")